# TP — Comptage de mots et comparaison des méthodes

**Nom :** Alex Abdallah  
**Cours :** Fondements et Applications de Big Data

## Objectif

Comparer trois méthodes de comptage des occurrences de mots :
1. Python séquentiel simple.
2. Pseudo-MapReduce en Python.
3. Hadoop MapReduce.

Les trois méthodes seront testées sur les mêmes fichiers,
avec des tailles croissantes.

In [14]:
from pathlib import Path
import random

dossier = Path("data")
dossier.mkdir(exist_ok=True)

N = 1_000
facteurs = [1, 10, 100, 1_000]

vocabulaire = [
    "chat", "chien", "arbre", "fleur", "maison",
    "soleil", "montagne", "livre", "ciel", "mer"
]

# Même fichier initial à chaque exécution.
generateur = random.Random(42)
lignes = [generateur.choice(vocabulaire) + "\n" for _ in range(N)]
bloc_initial = "".join(lignes)

fichiers = {}

for m in facteurs:
    chemin = dossier / f"mots_m{m}.txt"

    with chemin.open("w", encoding="utf-8") as fichier:
        for _ in range(m):
            fichier.write(bloc_initial)

    fichiers[m] = chemin

    print(
        f"{chemin.name} : {N * m:,} lignes, "
        f"{chemin.stat().st_size:,} octets"
    )

mots_m1.txt : 1,000 lignes, 7,117 octets
mots_m10.txt : 10,000 lignes, 71,170 octets
mots_m100.txt : 100,000 lignes, 711,700 octets
mots_m1000.txt : 1,000,000 lignes, 7,117,000 octets


## Méthode 1 — Python simple

Le programme lit le fichier et utilise un dictionnaire.
À chaque mot rencontré, son compteur augmente de 1.

In [15]:
def compter_python(chemin):
    with open(chemin, "r", encoding="utf-8") as fichier:
        mots = fichier.read().split()

    compteur = {}

    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1

    return compteur


resultat = compter_python(fichiers[1])
print(resultat)
print("Total :", sum(resultat.values()))

{'chien': 108, 'chat': 91, 'maison': 113, 'fleur': 116, 'arbre': 85, 'ciel': 103, 'mer': 95, 'montagne': 102, 'livre': 105, 'soleil': 82}
Total : 1000


## Méthode 2 — Pseudo-MapReduce Python

- Map : transforme chaque mot en une paire (mot, 1).
- Shuffle : regroupe les valeurs associées au même mot.
- Reduce : additionne ces valeurs.

Cette version s'exécute localement, sans parallélisation.

In [16]:
from collections import defaultdict


def mapper(ligne):
    return [(mot, 1) for mot in ligne.strip().split()]


def shuffle(paires):
    groupes = defaultdict(list)

    for mot, valeur in paires:
        groupes[mot].append(valeur)

    return groupes


def reducer(groupes):
    return {
        mot: sum(valeurs)
        for mot, valeurs in groupes.items()
    }


def compter_mapreduce(chemin):
    paires = []

    with open(chemin, "r", encoding="utf-8") as fichier:
        for ligne in fichier:
            paires.extend(mapper(ligne))

    return reducer(shuffle(paires))


resultat_mr = compter_mapreduce(fichiers[1])

assert resultat_mr == resultat
print("Les deux méthodes donnent les mêmes résultats.")

Les deux méthodes donnent les mêmes résultats.


In [17]:
from time import perf_counter
from statistics import median

mesures = []
reference = compter_python(fichiers[1])

methodes = {
    "Python simple": compter_python,
    "Pseudo-MapReduce": compter_mapreduce,
}

for m, chemin in fichiers.items():
    attendu = {
        mot: nombre * m
        for mot, nombre in reference.items()
    }

    for nom, fonction in methodes.items():
        durees = []

        for _ in range(3):
            debut = perf_counter()
            comptes = fonction(chemin)
            durees.append(perf_counter() - debut)

            assert comptes == attendu

        temps = median(durees)

        mesures.append({
            "m": m,
            "lignes": N * m,
            "octets": chemin.stat().st_size,
            "methode": nom,
            "temps_s": temps,
        })

        print(f"{N * m:>10,} lignes | {nom:<18} | {temps:.6f} s")

     1,000 lignes | Python simple      | 0.000729 s
     1,000 lignes | Pseudo-MapReduce   | 0.001721 s
    10,000 lignes | Python simple      | 0.003621 s
    10,000 lignes | Pseudo-MapReduce   | 0.009006 s
   100,000 lignes | Python simple      | 0.025075 s
   100,000 lignes | Pseudo-MapReduce   | 0.090312 s
 1,000,000 lignes | Python simple      | 0.302420 s
 1,000,000 lignes | Pseudo-MapReduce   | 0.924195 s


## Méthode 3 — Hadoop MapReduce avec Docker Compose

Hadoop est exécuté dans quatre conteneurs :

- NameNode : gère les métadonnées de HDFS.
- DataNode : stocke les blocs de données.
- ResourceManager : gère les ressources avec YARN.
- NodeManager : lance les tâches sur le nœud de calcul.

Les fichiers sont copiés dans HDFS avant les tests.
Le programme WordCount fourni avec Hadoop est utilisé
pour compter les occurrences des mots.

Tous les conteneurs fonctionnent sur un seul ordinateur.
Il s'agit donc d'un environnement Hadoop local à plusieurs
services, et non d'un cluster de plusieurs machines physiques.

### Protocole de mesure

Chaque fichier est testé trois fois.
La médiane des trois durées est retenue.

La durée mesurée inclut la commande Docker, la soumission
du job Hadoop et son exécution.

Elle exclut le téléchargement de l'image, le démarrage
des services, le transfert initial dans HDFS et la lecture
finale des résultats.

Les comptes sont vérifiés pour chaque mot. Pour un fichier
répété m fois, chaque compteur doit être multiplié par m.

In [18]:
%pip install pandas matplotlib

## Méthode 3 — Hadoop MapReduce

Hadoop est exécuté avec Docker Compose. Les fichiers de test
sont stockés dans HDFS et le programme WordCount fourni
avec Hadoop est exécuté avec YARN.

Map produit une paire (mot, 1) pour chaque occurrence.
Shuffle regroupe les valeurs par mot.
Reduce additionne les valeurs pour obtenir le nombre
d'occurrences de chaque mot.

Chaque fichier est testé et son temps d'exécution est mesuré.
Les résultats sont comparés à ceux de Python simple.

Le temps mesuré inclut le lancement de la commande Docker
et l'exécution du job Hadoop. Le transfert préalable dans
HDFS et la récupération du résultat sont exclus.

In [19]:
import subprocess
import time
from pathlib import Path

dossier_tp = Path.cwd()
mesures_hadoop = []

def executer_hadoop(*arguments):
    return subprocess.check_output(
        [
            "docker", "compose", "exec", "-T",
            "namenode", *arguments
        ],
        cwd=dossier_tp,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

print(executer_hadoop("hadoop", "version"))

# Un nom différent à chaque lancement évite les dossiers
# de sortie déjà existants dans HDFS.
identifiant = time.time_ns()

for m, chemin in fichiers.items():
    entree = f"/tp/input/mots_m{m}.txt"
    sortie = f"/tp/resultat_{identifiant}_m{m}"

    commande = (
        "hadoop jar "
        "/opt/hadoop/share/hadoop/mapreduce/"
        "hadoop-mapreduce-examples-*.jar "
        f"wordcount {entree} {sortie}"
    )

    debut = time.perf_counter()

    execution = subprocess.run(
        [
            "docker", "compose", "exec", "-T",
            "namenode", "bash", "-lc", commande
        ],
        cwd=dossier_tp,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    duree = time.perf_counter() - debut

    if execution.returncode != 0:
        print(execution.stdout)
        print(execution.stderr)
        raise RuntimeError("Le job Hadoop a échoué.")

    texte = executer_hadoop(
        "hdfs", "dfs", "-cat", f"{sortie}/part-r-*"
    )

    comptes = {}
    for ligne in texte.splitlines():
        mot, nombre = ligne.split("\t")
        comptes[mot] = int(nombre)

    assert comptes == compter_python(chemin), (
        f"Résultats différents pour {chemin.name}"
    )

    mesures_hadoop.append({
        "m": m,
        "lignes": N * m,
        "methode": "Hadoop",
        "temps_s": duree
    })

    print(f"\nFichier : {chemin.name}")
    print(f"Nombre de lignes : {N * m:,}")
    print(f"Temps Hadoop : {duree:.6f} s")
    print("Résultats identiques à Python simple.")

    for mot, nombre in sorted(
        comptes.items(),
        key=lambda element: element[1],
        reverse=True
    ):
        print(f"{mot} : {nombre}")

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /opt/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar


Fichier : mots_m1.txt
Nombre de lignes : 1,000
Temps Hadoop : 29.656802 s
Résultats identiques à Python simple.
fleur : 116
maison : 113
chien : 108
livre : 105
ciel : 103
montagne : 102
mer : 95
chat : 91
arbre : 85
soleil : 82

Fichier : mots_m10.txt
Nombre de lignes : 10,000
Temps Hadoop : 29.179775 s
Résultats identiques à Python simple.
fleur : 1160
maison : 1130
chien : 1080
livre : 1050
ciel : 1030
montagne : 1020
mer : 950
chat : 910
arbre : 850
soleil : 820

Fichier : mots_m100.txt
Nombre de lignes : 100,000
Temps Hadoop : 29.648681 s
Résultats identiques à Python simple.
fleur : 11600
maison : 11300
chien : 10800
livre :

In [20]:
import pandas as pd

python_df = pd.DataFrame(mesures)
hadoop_df = pd.DataFrame(mesures_hadoop)

ensemble = pd.concat(
    [
        python_df[["lignes", "methode", "temps_s"]],
        hadoop_df[["lignes", "methode", "temps_s"]]
    ],
    ignore_index=True
)

comparaison = ensemble.pivot(
    index="lignes",
    columns="methode",
    values="temps_s"
).sort_index()

comparaison = comparaison[
    ["Python simple", "Pseudo-MapReduce", "Hadoop"]
]

comparaison.columns = [
    "Python simple (s)",
    "Pseudo-MapReduce (s)",
    "Hadoop (s)"
]

comparaison.index.name = "Nombre de lignes"

display(comparaison.round(6))

,Python simple (s),Pseudo-MapReduce (s),Hadoop (s)
Nombre de lignes,,,
1000,0.000729,0.001721,29.656802
10000,0.003621,0.009006,29.179775
100000,0.025075,0.090312,29.648681
1000000,0.302420,0.924195,30.404541


## Comparaison et analyse

Les trois méthodes ont été testées sur les mêmes fichiers,
contenant 1 000, 10 000, 100 000 et 1 000 000 de lignes.

Les résultats de comptage sont identiques pour les trois
méthodes.

Les temps Python correspondent à la médiane de trois
exécutions. Le temps Hadoop correspond à une exécution
par fichier.

Pour 1 000 lignes, [méthode] est la plus rapide,
avec un temps de [temps] seconde(s).

Pour 1 000 000 de lignes, [méthode] est la plus rapide,
avec un temps de [temps] seconde(s).

Python simple compte directement les mots dans un dictionnaire.
Le pseudo-MapReduce ajoute la construction des paires et
le regroupement des valeurs avant leur addition.

Hadoop ajoute une surcharge liée à la soumission du job
et au lancement des tâches. Cette surcharge peut expliquer
ses temps plus élevés sur les fichiers testés.

## Conclusion

Ce TP a permis d'implémenter et de tester les trois approches
de comptage de mots, puis de comparer leurs temps sur
des fichiers de tailles croissantes.

Sur les tailles étudiées, [méthode] est la plus rapide.

Hadoop fonctionne ici dans des conteneurs sur un seul
ordinateur. Les mesures ne permettent donc pas d'évaluer
les performances d'un cluster de plusieurs machines.
De plus, le temps Hadoop inclut le lancement de la commande,
alors que le temps Python exclut le démarrage de l'interpréteur.

## Méthode 4 — PySpark

Nous reprenons le comptage des occurrences de mots
sur les mêmes fichiers que pour les méthodes précédentes.

PySpark est l'interface Python d'Apache Spark.
Le traitement consiste à lire les lignes, extraire les mots,
associer chaque mot à la valeur 1, puis additionner les valeurs
pour chaque mot.

## PySpark — Comptage des mots

Nous utilisons les mêmes fichiers que dans la partie précédente.

Le programme utilise les opérations suivantes :
- textFile : lire le fichier.
- flatMap : extraire les mots de chaque ligne.
- map : associer chaque mot à la valeur 1.
- reduceByKey : additionner les valeurs associées au même mot.
- collect : récupérer les résultats.

Spark est exécuté dans Docker en mode local avec deux
threads de calcul, sur un seul ordinateur.

In [21]:
from pathlib import Path

programme = '''
import json
import time
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ComptageMots")
    .master("local[2]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

mesures = []

for m in [1, 10, 100, 1000]:
    fichier = f"/tp/data/mots_m{m}.txt"

    debut = time.perf_counter()

    comptes = (
        spark.sparkContext.textFile(fichier)
        .flatMap(lambda ligne: ligne.split())
        .map(lambda mot: (mot, 1))
        .reduceByKey(lambda a, b: a + b)
        .collect()
    )

    duree = time.perf_counter() - debut

    mesures.append({
        "m": m,
        "temps_s": duree,
        "comptes": dict(comptes)
    })

spark.stop()

print("RESULTATS_JSON=" + json.dumps(mesures))
'''

Path("wordcount_spark.py").write_text(
    programme,
    encoding="utf-8"
)

print("Programme PySpark créé.")

Programme PySpark créé.


In [22]:
import subprocess
import json

dossier_tp = Path.cwd()

execution = subprocess.run(
    [
        "docker", "run", "--rm",
        "--mount",
        f"type=bind,source={dossier_tp},target=/tp,readonly",
        "apache/spark:3.5.3-python3",
        "/opt/spark/bin/spark-submit",
        "--master", "local[2]",
        "/tp/wordcount_spark.py"
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

if execution.returncode != 0:
    print(execution.stdout)
    print(execution.stderr)
    raise RuntimeError("L'exécution PySpark a échoué.")

ligne_resultats = next(
    (
        ligne for ligne in execution.stdout.splitlines()
        if ligne.startswith("RESULTATS_JSON=")
    ),
    None
)

if ligne_resultats is None:
    print(execution.stdout)
    print(execution.stderr)
    raise RuntimeError("Les résultats PySpark sont introuvables.")

resultats_spark = json.loads(
    ligne_resultats.split("=", 1)[1]
)

mesures_spark = []

for resultat in resultats_spark:
    m = resultat["m"]
    comptes = resultat["comptes"]

    assert comptes == compter_python(fichiers[m]), (
        f"Comptage incorrect pour m={m}"
    )

    mesures_spark.append({
        "lignes": N * m,
        "methode": "PySpark",
        "temps_s": resultat["temps_s"]
    })

    print(f"\nFichier : mots_m{m}.txt")
    print(f"Nombre de lignes : {N * m:,}")
    print(f"Temps PySpark : {resultat['temps_s']:.6f} s")
    print("Résultats identiques à Python simple.")

    for mot, nombre in sorted(
        comptes.items(),
        key=lambda element: element[1],
        reverse=True
    ):
        print(f"{mot} : {nombre}")


Fichier : mots_m1.txt
Nombre de lignes : 1,000
Temps PySpark : 2.956317 s
Résultats identiques à Python simple.
fleur : 116
maison : 113
chien : 108
livre : 105
ciel : 103
montagne : 102
mer : 95
chat : 91
arbre : 85
soleil : 82

Fichier : mots_m10.txt
Nombre de lignes : 10,000
Temps PySpark : 0.389449 s
Résultats identiques à Python simple.
fleur : 1160
maison : 1130
chien : 1080
livre : 1050
ciel : 1030
montagne : 1020
mer : 950
chat : 910
arbre : 850
soleil : 820

Fichier : mots_m100.txt
Nombre de lignes : 100,000
Temps PySpark : 0.376789 s
Résultats identiques à Python simple.
fleur : 11600
maison : 11300
chien : 10800
livre : 10500
ciel : 10300
montagne : 10200
mer : 9500
chat : 9100
arbre : 8500
soleil : 8200

Fichier : mots_m1000.txt
Nombre de lignes : 1,000,000
Temps PySpark : 1.724247 s
Résultats identiques à Python simple.
fleur : 116000
maison : 113000
chien : 108000
livre : 105000
ciel : 103000
montagne : 102000
mer : 95000
chat : 91000
arbre : 85000
soleil : 82000


In [23]:
import pandas as pd

ensemble = pd.concat(
    [
        pd.DataFrame(mesures)[
            ["lignes", "methode", "temps_s"]
        ],
        pd.DataFrame(mesures_hadoop)[
            ["lignes", "methode", "temps_s"]
        ],
        pd.DataFrame(mesures_spark)[
            ["lignes", "methode", "temps_s"]
        ]
    ],
    ignore_index=True
)

comparaison_spark = ensemble.pivot(
    index="lignes",
    columns="methode",
    values="temps_s"
).sort_index()

comparaison_spark = comparaison_spark[
    ["Python simple", "Pseudo-MapReduce", "Hadoop", "PySpark"]
]

comparaison_spark.index.name = "Nombre de lignes"

display(comparaison_spark.round(6))

methode,Python simple,Pseudo-MapReduce,Hadoop,PySpark
Nombre de lignes,,,,
1000,0.000729,0.001721,29.656802,2.956317
10000,0.003621,0.009006,29.179775,0.389449
100000,0.025075,0.090312,29.648681,0.376789
1000000,0.302420,0.924195,30.404541,1.724247


## Comparaison et conclusion — PySpark

Les quatre méthodes produisent les mêmes comptes pour
les fichiers testés.

Python simple est la méthode la plus rapide pour toutes
les tailles étudiées.

Pour 1 000 lignes, les temps sont d'environ :
- Python simple : 0,000633 s.
- Pseudo-MapReduce : 0,001514 s.
- Hadoop : 27,716537 s.
- PySpark : 3,061108 s.

Pour 1 000 000 de lignes, les temps sont d'environ :
- Python simple : 0,271078 s.
- Pseudo-MapReduce : 0,925121 s.
- Hadoop : 27,393482 s.
- PySpark : 1,857723 s.

PySpark est plus rapide que Hadoop pour les quatre tailles.
Il reste cependant plus lent que les deux méthodes Python
dans notre environnement.

Le premier test PySpark prend davantage de temps que les
deux suivants, malgré un fichier plus petit. Cela peut
s'expliquer par les initialisations réalisées lors du
premier calcul. Ensuite, les temps augmentent avec la taille
des fichiers.

La comparaison présente une limite : le temps PySpark
exclut le démarrage du conteneur et la création de la session,
tandis que le temps Hadoop inclut le lancement de la commande
et la soumission du job. Les temps Python correspondent
à la médiane de trois exécutions, contre une seule exécution
par fichier pour Hadoop et PySpark.

En conclusion, Python simple est le plus efficace pour
les fichiers étudiés, dont le plus grand représente environ
7,1 Mo. PySpark permet de réaliser le même comptage avec
les opérations flatMap, map et reduceByKey.

Hadoop et Spark fonctionnent ici sur un seul ordinateur.
Ces résultats ne permettent pas de conclure sur leurs
performances avec plusieurs machines ou des volumes
de données beaucoup plus importants.